# Reproducción de Wei et al. (2021): QCNN con filtro LCU en MNIST y Fashion-MNIST

Objetivo: comprobar que el QCNN de Wei, Chen, Zhou y Long (2021), *"A Quantum Convolutional Neural Network on NISQ Devices"* (arXiv:2104.06918v3), se acerca a la exactitud publicada (96.3% sin ruido) con el protocolo de entrenamiento de este benchmark.

La lógica reutilizable está en `src/qcnn_benchmark/` y este notebook solo configura y ejecuta. La versión exploratoria original está en `notebooks/dev/02_reproduce_wei.ipynb`, donde se explica por qué la implementación de qml-benchmarks de Xanadu no es fiel al paper y por qué el circuito se reconstruyó a partir de sus ecuaciones. El detalle del circuito está en `src/qcnn_benchmark/models/qcnn_wei.py`.

El notebook corre dos tareas. MNIST 1 vs 8 es la que reporta 96.3% en la Tabla I del paper (sin ruido) y sirve como validación. Fashion-MNIST camiseta vs. pantalón se incluye como referencia adicional, ya que Wei et al. no evalúan Fashion-MNIST.

Nota sobre los datasets: camiseta vs. pantalón (T-shirt/top vs. Trouser) no forma parte de los experimentos de E0A en adelante, que usan Fashion-MNIST coat vs. shirt, MNIST 4 vs. 9 y MNIST 1 vs. 0. La corrida MNIST 1 vs 8 sigue siendo la validación contra el paper.

In [ ]:
import os

import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())  # evita errores de certificado SSL con el Python de python.org en macOS

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from qcnn_benchmark.data import load_mnist_pool, load_fashion_mnist_pool
from qcnn_benchmark.representations import build_amplitude_dataset
from qcnn_benchmark.models import qcnn_wei
from qcnn_benchmark.training import train_binary_classifier, normal_init
from qcnn_benchmark.metrics import batch_accuracy, predict_labels

print("Parámetros entrenables (9 filtro + 37 hamiltoniano):", qcnn_wei.TOTAL_PARAMS, "(Tabla I del paper: 46)")


## 1. Carga de datos y representación (codificación de amplitud, 1024 = 32×32)

Muestreo estratificado balanceado, igual que en el notebook de Hur, con 500 ejemplos de entrenamiento, 250 de validación y 500 de prueba por clase y semilla `20260802`. No se usa PCA. Cada imagen de 28×28 se rellena con ceros a 32×32, se aplana a 1024 valores y se normaliza en norma L2 (ver `qcnn_benchmark.representations.amplitude`).

In [ ]:
X_MNIST_ALL, Y_MNIST_ALL = load_mnist_pool(normalize=False)
print("Pool MNIST:", X_MNIST_ALL.shape, Y_MNIST_ALL.shape)

print("MNIST 1 vs 8:")
rep_1v8 = build_amplitude_dataset(X_MNIST_ALL, Y_MNIST_ALL, class_pos=1, class_neg=8)


## 2. Entrenamiento

Mismo protocolo que `00_reproduce_hur.ipynb` (`qcnn_benchmark.training.train_binary_classifier`): BCE, Adam (lr=0.01, β1=0.9, β2=0.999), 200 actualizaciones, lote de 25, recorte de la norma global del gradiente a 5.0, early stopping con paciencia de 5 chequeos de validación cada 10 actualizaciones (δ=1e-4) y selección del checkpoint con menor pérdida de validación. La única diferencia es la inicialización de pesos, que es $\mathcal{N}(0, 0.1)$ en lugar de uniforme (`qcnn_benchmark.training.normal_init`).

Se usa una sola semilla (`RUN_SEED = 0`), porque el objetivo es validar que el circuito LCU con Hamiltoniano reproduce el número publicado. Los experimentos usan 5 semillas.

In [ ]:
RUN_SEED = 0


def plot_loss_curve(result, title, extra_vline=None, extra_vline_label=""):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, result["n_updates_run"] + 1), result["train_loss_history"],
            label="Pérdida de entrenamiento (por actualización)", alpha=0.7)
    val_updates, val_losses = zip(*result["val_loss_history"])
    ax.plot(val_updates, val_losses, "o-", label="Pérdida de validación (cada 10 actualizaciones)", color="darkorange")
    if result["stopped_early_at"]:
        ax.axvline(result["stopped_early_at"], color="red", linestyle="--", alpha=0.6, label="Early stopping")
    if extra_vline is not None:
        ax.axvline(extra_vline, color="gray", linestyle=":", alpha=0.7, label=extra_vline_label)
    ax.set_xlabel("Actualización de parámetros")
    ax.set_ylabel("Entropía cruzada binaria")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, title, class_labels):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(class_labels)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(class_labels)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta real")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()
    print(cm)
    return cm


def report_accuracy(train_acc, test_acc, tag, target_acc=None):
    print("=" * 70)
    print(f"{tag}, Wei et al. QCNN, 1 semilla de ejecución")
    print("=" * 70)
    print(f"Exactitud de entrenamiento: {train_acc * 100:.2f}%")
    print(f"Exactitud de prueba:        {test_acc * 100:.2f}%")
    if target_acc is None:
        print("Sin objetivo publicado por Wei et al. para esta tarea.")
        return
    gap_pp = test_acc * 100 - target_acc
    print(f"Objetivo publicado (Tabla I, sin ruido): {target_acc}%")
    print(f"Diferencia (prueba - objetivo): {gap_pp:+.2f} puntos porcentuales")
    if abs(gap_pp) <= 1.0:
        print("-> Dentro de ~1pp del objetivo publicado: fidelidad razonable para 1 semilla.")
    elif abs(gap_pp) <= 3.0:
        print("-> Brecha moderada (1-3pp). Aceptable para 1 semilla, revisar antes de 5 semillas.")
    else:
        print("-> Brecha grande (>3pp). Revisar antes de escalar a 5 semillas.")


## 3. Corrida A: MNIST 1 vs 8 (referencia 96.3%)

Convención de etiquetas: $y=1$ para el dígito "1" y $y=0$ para el dígito "8". El paper usa la convención opuesta, pero el problema es simétrico y la exactitud alcanzable no cambia.

In [ ]:
result_1v8 = train_binary_classifier(
    qcnn_wei.predict_proba, qcnn_wei.TOTAL_PARAMS, rep_1v8, normal_init,
    run_seed=RUN_SEED, tag="1v8",
)
plot_loss_curve(result_1v8, "MNIST 1 vs 8, Wei et al. QCNN, curva de pérdida")


In [ ]:
params_1v8 = result_1v8["params"]
train_acc_1v8 = batch_accuracy(qcnn_wei.predict_proba, params_1v8, rep_1v8["X_train"], rep_1v8["y_train"])
test_acc_1v8 = batch_accuracy(qcnn_wei.predict_proba, params_1v8, rep_1v8["X_test"], rep_1v8["y_test"])
report_accuracy(train_acc_1v8, test_acc_1v8, "MNIST 1 vs 8", target_acc=96.3)


In [ ]:
y_pred_test_1v8 = predict_labels(qcnn_wei.predict_proba, params_1v8, rep_1v8["X_test"])
_ = plot_confusion(rep_1v8["y_test"], y_pred_test_1v8,
                    "Matriz de confusión, MNIST 1 vs 8 (prueba)",
                    ["0 (dígito 8)", "1 (dígito 1)"])


## 3.1 Diagnóstico opcional: convergencia con más actualizaciones

En la corrida exploratoria original (`notebooks/dev/02_reproduce_wei.ipynb`), 200 actualizaciones dieron 94.80%, 1.5 puntos por debajo del 96.3% publicado, y la pérdida de validación siguió mejorando en los 20 chequeos. Eso sugiere que el circuito, con 46 parámetros, no alcanza a converger en 200 actualizaciones. Esta celda repite la corrida con 500 actualizaciones para comprobarlo. Es un diagnóstico adicional y no reemplaza la corrida de la sección 3, que sigue el protocolo del benchmark.

In [ ]:
result_1v8_extended = train_binary_classifier(
    qcnn_wei.predict_proba, qcnn_wei.TOTAL_PARAMS, rep_1v8, normal_init,
    run_seed=RUN_SEED, n_updates=500, tag="1v8-extended-500",
)

params_1v8_ext = result_1v8_extended["params"]
train_acc_1v8_ext = batch_accuracy(qcnn_wei.predict_proba, params_1v8_ext, rep_1v8["X_train"], rep_1v8["y_train"])
test_acc_1v8_ext = batch_accuracy(qcnn_wei.predict_proba, params_1v8_ext, rep_1v8["X_test"], rep_1v8["y_test"])
gap_pp_ext = test_acc_1v8_ext * 100 - 96.3
gap_pp = test_acc_1v8 * 100 - 96.3

print("=" * 70)
print("DIAGNOSTICO: MNIST 1 vs 8, misma semilla, 500 actualizaciones en vez de 200")
print("=" * 70)
print(f"Exactitud de prueba: {test_acc_1v8_ext * 100:.2f}%  (200 updates: {test_acc_1v8 * 100:.2f}%)")
print(f"Diferencia vs. objetivo: {gap_pp_ext:+.2f}pp  (200 updates: {gap_pp:+.2f}pp)")
if abs(gap_pp_ext) < abs(gap_pp):
    print("-> La brecha se REDUJO con más actualizaciones: consistente con falta de convergencia a 200 updates.")
else:
    print("-> La brecha NO se redujo (o empeoró): revisar la implementación antes de escalar a 5 semillas.")

plot_loss_curve(result_1v8_extended, "MNIST 1 vs 8, diagnóstico de convergencia (500 actualizaciones)",
                 extra_vline=200, extra_vline_label="Fin de la corrida oficial (200 updates)")


## 4. Corrida B: Fashion-MNIST camiseta vs. pantalón (referencia adicional)

Wei et al. no evalúan Fashion-MNIST, así que no hay un valor publicado con el cual comparar. Se usa la misma codificación de amplitud. Convención de etiquetas: $y=1$ para "T-shirt/top" (camiseta, clase 0 de Fashion-MNIST) y $y=0$ para "Trouser" (pantalón, clase 1). Esta tarea no forma parte de los experimentos posteriores.

In [ ]:
X_FASHION_ALL, Y_FASHION_ALL = load_fashion_mnist_pool(normalize=False)
print("Pool Fashion-MNIST:", X_FASHION_ALL.shape, Y_FASHION_ALL.shape)
print("0 = T-shirt/top (camiseta), 1 = Trouser (pantalón)")

rep_shirt_trouser = build_amplitude_dataset(X_FASHION_ALL, Y_FASHION_ALL, class_pos=0, class_neg=1)

result_shirt_trouser = train_binary_classifier(
    qcnn_wei.predict_proba, qcnn_wei.TOTAL_PARAMS, rep_shirt_trouser, normal_init,
    run_seed=RUN_SEED, tag="shirt-trouser",
)
plot_loss_curve(result_shirt_trouser, "Fashion-MNIST camiseta vs. pantalón, Wei et al. QCNN, curva de pérdida")


In [ ]:
params_shirt_trouser = result_shirt_trouser["params"]
train_acc_st = batch_accuracy(qcnn_wei.predict_proba, params_shirt_trouser, rep_shirt_trouser["X_train"], rep_shirt_trouser["y_train"])
test_acc_st = batch_accuracy(qcnn_wei.predict_proba, params_shirt_trouser, rep_shirt_trouser["X_test"], rep_shirt_trouser["y_test"])
report_accuracy(train_acc_st, test_acc_st, "Fashion-MNIST camiseta vs. pantalón")


In [ ]:
y_pred_test_st = predict_labels(qcnn_wei.predict_proba, params_shirt_trouser, rep_shirt_trouser["X_test"])
_ = plot_confusion(rep_shirt_trouser["y_test"], y_pred_test_st,
                    "Matriz de confusión, Fashion-MNIST camiseta/pantalón (prueba)",
                    ["0 (pantalón)", "1 (camiseta)"])


## 5. Resumen

| Tarea | Exactitud entrenamiento | Exactitud prueba | Objetivo Wei et al. | Comparable |
|---|---|---|---|---|
| MNIST 1 vs 8 | ver celda de resultados | ver celda de resultados | 96.3% (Tabla I, sin ruido) | Sí |
| Fashion-MNIST camiseta vs. pantalón | ver celda de resultados | ver celda de resultados | No publicado | No |

Notas:

- La post-selección de las 4 ancillas se implementa como combinación lineal directa, que es exacta en el caso sin ruido (ver `qcnn_benchmark.models.qcnn_wei`). Para shots y ruido, `qcnn_benchmark.execution.shots` fija el filtro LCU y solo entrena el Hamiltoniano de lectura.
- E0A usa los tres datasets de los experimentos (Fashion-MNIST coat vs. shirt, MNIST 4 vs. 9 y MNIST 1 vs. 0) y no las tareas de este notebook.